# S1 equities — Monte Carlo EV vs SPY

Sealed OOS net returns only. **This notebook is EV vs SPY** (expected value, HAC/bootstrap significance of the mean, $P(\mathrm{not\ beat\ SPY})$). It does not compute prop-firm pass rates.


## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.monte_carlo.loaders import (
    aligned_strategy_spy,
    find_repo_root,
    load_sealed_s1,
)
from risk.monte_carlo.report import run_ev_vs_spy

ROOT = find_repo_root(ROOT)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("ROOT", ROOT)

SLEEVE = 's1'
BAR = 'W'
PERIODS_PER_YEAR = 52
DEFAULT_H = 52
DEFAULT_BLOCK = 8.0
DEFAULT_N_SIM = 400


## 1. Data Loading


In [ ]:
FRAME = aligned_strategy_spy(load_sealed_s1(ROOT), bar=BAR)
print(FRAME.tail())
print("n_bars", len(FRAME), "start", FRAME.index.min().date(), "end", FRAME.index.max().date())


## 2. Historical EV & significance

Headline: **t-stat, p-value, `ci_excludes_zero`**. These describe the *historical mean* of sealed OOS period returns (H0: $E[r]=0$), not a single simulated path. Block-bootstrap $P^*(\hat\mu^*\le 0)$ is the nonparametric counterpart. PSR is secondary Sharpe quality, not the EV-significance metric.


In [ ]:
from risk.monte_carlo.ev_stats import ev_significance, excess_returns
from risk.monte_carlo.plots import significance_frame

hist = ev_significance(
    FRAME["strategy"],
    periods_per_year=PERIODS_PER_YEAR,
    mean_block_length=DEFAULT_BLOCK,
    n_bootstrap=800,
    random_seed=0,
)
excess = ev_significance(
    excess_returns(FRAME["strategy"], FRAME["spy"]),
    periods_per_year=PERIODS_PER_YEAR,
    mean_block_length=DEFAULT_BLOCK,
    n_bootstrap=800,
    random_seed=0,
)
print("Strategy EV significance (historical mean)")
display(significance_frame(hist))
print("Excess vs SPY EV significance (distinct from P(not beat SPY))")
display(significance_frame(excess))


## 3. Joint path simulator

Stationary block bootstrap draws **paired** strategy and SPY paths (same block indices). Independent resampling would invalidate $P(\mathrm{not\ beat\ SPY})$. Leverage $k$ scales **strategy** simple returns only (`r' = k r`); it is a risk-budget overlay, not a re-run of sleeve vol-targeting.


## 4. Interactive horizon / leverage

Gold/red lines are the **sealed OOS** wealth over the last (and, if the sample is longer, first) $H$ bars, drawn on the bootstrap fan. The excess fan is $W_{\mathrm{strat}}/W_{\mathrm{SPY}}$.


In [ ]:
PACK = {}

def run(horizon, leverage, n_simulations, mean_block_length, haircut_bps):
    pack = run_ev_vs_spy(
        FRAME,
        n_simulations=int(n_simulations),
        horizon=int(horizon),
        leverage=float(leverage),
        mean_block_length=float(mean_block_length),
        periods_per_year=PERIODS_PER_YEAR,
        random_seed=0,
        n_bootstrap=N_BOOTSTRAP,
        haircut_bps=float(haircut_bps),
    )
    PACK.clear()
    PACK.update(pack)
    print("Headline (no prop-firm pass rates)")
    display(pack["headline"].to_frame("value"))
    print("Historical strategy EV significance")
    display(pack["hist_table"])
    print("Excess vs SPY (expectation), separate from P(not beat SPY)")
    display(pack["excess_table"])
    print("Pathwise holes (max DD / time in hole / recover)")
    display(pack["holes_summary"].to_frame("value"))
    print("EV concentration (mean vs median vs CVaR; top-decile share)")
    display(pack["concentration"].to_frame("value"))
    print("Joint shape vs SPY")
    display(pack["joint_shape"].to_frame("value"))
    display(pack["fan"])
    display(pack["excess_fan"])
    display(pack["max_dd_hist"])
    display(pack["dd_scatter"])
    display(pack["terminals"])
    return pack

N_BOOTSTRAP = 600
pack = run(DEFAULT_H, 1.0, DEFAULT_N_SIM, DEFAULT_BLOCK, 0.0)
try:
    import ipywidgets as w
    ui = w.interactive(
        run,
        horizon=w.IntSlider(min=8, max=max(DEFAULT_H * 2, 16), value=DEFAULT_H, step=1, description="H"),
        leverage=w.FloatSlider(min=0.25, max=3.0, value=1.0, step=0.25, description="k"),
        n_simulations=w.IntSlider(min=50, max=2000, value=DEFAULT_N_SIM, step=50, description="n_sim"),
        mean_block_length=w.FloatSlider(min=2.0, max=30.0, value=DEFAULT_BLOCK, step=1.0, description="block L"),
        haircut_bps=w.FloatSlider(min=0.0, max=5.0, value=0.0, step=0.25, description="haircut bps"),
    )
    display(ui)
except Exception as exc:
    print("ipywidgets unavailable (%s); default run already executed" % exc)


## 5. Pathwise holes

Not $P(\mathrm{ever\ underwater})$. Max-DD distribution, time spent below the peak, bars to recover from the trough, and the scatter of **terminal wealth vs max DD** (is $E[W_H]$ bought with a deep hole?). Top-decile EV share flags a fragile right tail.


In [ ]:
if not PACK:
    raise RuntimeError("run() did not populate PACK")
print("holes head")
display(PACK["holes"].head())
display(PACK["holes_summary"].to_frame("value"))
display(PACK["concentration"].to_frame("value"))
display(PACK["max_dd_hist"])
display(PACK["dd_scatter"])


## 6. Joint shape vs SPY

Same paired bootstrap columns as $P(\mathrm{not\ beat\ SPY})$. Median pathwise beta/corr, down-market capture (mean strategy return on bars with SPY $<0$), and $P(W_s\le W_{\mathrm{spy}}\mid W_{\mathrm{spy}}<1)$. Excess-wealth fan is $W_s/W_{\mathrm{spy}}$.


In [ ]:
display(PACK["joint_shape"].to_frame("value"))
display(PACK["excess_fan"])
print("OOS terminal percentile among simulated paths", PACK["headline"]["oos_terminal_percentile"])


## 7. Evaluation

- Historical $E[r]$ with HAC t/p and `ci_excludes_zero`
- Bootstrap $P^*(\mu\le 0)$
- Horizon $E[W_H-1]$ vs median vs CVaR; top-decile EV share
- Pathwise max DD (median / 5th percentile) and terminal-vs-DD scatter
- Excess-wealth fan and $P(\mathrm{not\ beat\ SPY})$ plus down-market capture
- Sealed OOS overlay on the fan; OOS terminal percentile vs the storm

S1 sealed series is **weekly**. SPY is compounded onto the same Monday–Monday weeks.


In [ ]:
# Optional HMM (univariate strategy only — not for P(beat SPY))
from risk.monte_carlo.hmm_simulator import GaussianHMMSimulator
from risk.monte_carlo.ev_stats import horizon_ev, cvar, terminal_simple_return

hmm = GaussianHMMSimulator(n_simulations=200, random_seed=0)
hmm.fit(FRAME["strategy"])
hmm_paths = hmm.simulate(DEFAULT_H)
print(hmm.summary(hmm_paths))
print("HMM horizon EV (strategy only)", horizon_ev(hmm_paths))
print("HMM CVaR5", cvar(terminal_simple_return(hmm_paths), alpha=0.05))
